In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/flights.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (336776, 21)


,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,...,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,name
0,0,2013,1,1,517.0,515,2.0,830.0,819,11.0,...,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00,United Air Lines Inc.
1,1,2013,1,1,533.0,529,4.0,850.0,830,20.0,...,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00,United Air Lines Inc.
2,2,2013,1,1,542.0,540,2.0,923.0,850,33.0,...,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00,American Airlines Inc.
3,3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,...,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01 05:00:00,JetBlue Airways
4,4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,...,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01 06:00:00,Delta Air Lines Inc.


In [2]:
df = df.dropna(subset=["arr_delay"]).copy()

df["delayed"] = (df["arr_delay"] >= 15).astype(int)

print("Labeled flights:", len(df))
print("\nTarget distribution:")
display(df["delayed"].value_counts())
print("\nDelay rate:", round(df["delayed"].mean() * 100, 2), "%")

Labeled flights: 327346

Target distribution:


delayed
0    247246
1     80100
Name: count, dtype: int64


Delay rate: 24.47 %


In [3]:
df["scheduled_datetime"] = pd.to_datetime(
    df["time_hour"], errors="coerce"
) + pd.to_timedelta(
    df["minute"], unit="m"
) - pd.to_timedelta(
    pd.to_datetime(df["time_hour"], errors="coerce").dt.minute,
    unit="m"
)

df["day_of_week"] = df["scheduled_datetime"].dt.dayofweek
df["day_of_year"] = df["scheduled_datetime"].dt.dayofyear

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["day_of_year_sin"] = np.sin(
    2 * np.pi * df["day_of_year"] / 365.25
)
df["day_of_year_cos"] = np.cos(
    2 * np.pi * df["day_of_year"] / 365.25
)

display(df[[
    "scheduled_datetime", "hour", "minute", "day_of_week",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos"
]].head())

/var/folders/42/1195kj1s5cnfdl1h85ykvth40000gn/T/ipykernel_62168/1990143060.py:10: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["day_of_week"] = df["scheduled_datetime"].dt.dayofweek
/var/folders/42/1195kj1s5cnfdl1h85ykvth40000gn/T/ipykernel_62168/1990143060.py:11: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  df["day_of_year"] = df["scheduled_datetime"].dt.dayofyear


,scheduled_datetime,hour,minute,day_of_week,hour_sin,hour_cos,dow_sin,dow_cos
0,2013-01-01 05:15:00,5,15,1,0.965926,2.588190e-01,0.781831,0.62349
1,2013-01-01 05:29:00,5,29,1,0.965926,2.588190e-01,0.781831,0.62349
2,2013-01-01 05:40:00,5,40,1,0.965926,2.588190e-01,0.781831,0.62349
3,2013-01-01 05:45:00,5,45,1,0.965926,2.588190e-01,0.781831,0.62349
4,2013-01-01 06:00:00,6,0,1,1.000000,6.123234e-17,0.781831,0.62349


In [4]:
feature_cols = [
    "month",
    "day",
    "hour",
    "minute",
    "day_of_week",
    "day_of_year",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    "distance",
    "carrier",
    "origin",
    "dest",
]

X = df[feature_cols].copy()
y = df["delayed"].copy()

print("Feature columns:")
print(X.columns.tolist())
print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)

Feature columns:
['month', 'day', 'hour', 'minute', 'day_of_week', 'day_of_year', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'day_of_year_sin', 'day_of_year_cos', 'distance', 'carrier', 'origin', 'dest']

Features shape: (327346, 16)
Target shape: (327346,)


In [5]:
ordered_indices = df["scheduled_datetime"].sort_values().index

X = X.loc[ordered_indices].reset_index(drop=True)
y = y.loc[ordered_indices].reset_index(drop=True)
dates = df.loc[ordered_indices, "scheduled_datetime"].reset_index(drop=True)

n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

print("Train:", X_train.shape, dates.iloc[0], "to", dates.iloc[train_end - 1])
print("Validation:", X_val.shape, dates.iloc[train_end], "to", dates.iloc[val_end - 1])
print("Test:", X_test.shape, dates.iloc[val_end], "to", dates.iloc[-1])

Train: (229142, 16) 2013-01-01 05:15:00 to 2013-09-13 19:05:00
Validation: (49102, 16) 2013-09-13 19:05:00 to 2013-11-06 13:00:00
Test: (49102, 16) 2013-11-06 13:00:00 to 2013-12-31 23:59:00


In [6]:
summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Rows": [len(y_train), len(y_val), len(y_test)],
    "Delay rate (%)": [
        y_train.mean() * 100,
        y_val.mean() * 100,
        y_test.mean() * 100
    ]
})

display(summary.round(2))

assert len(X_train) + len(X_val) + len(X_test) == len(X)
assert X_train.index.max() < X_val.index.min()
assert X_val.index.max() < X_test.index.min()

print("Chronological split checks passed.")

,Split,Rows,Delay rate (%)
0,Train,229142,25.94
1,Validation,49102,15.39
2,Test,49102,26.69


Chronological split checks passed.


In [7]:
forbidden_features = {
    "dep_time", "dep_delay", "arr_time", "arr_delay",
    "air_time", "arr_delay", "delayed"
}

assert forbidden_features.isdisjoint(X.columns)

print("No forbidden outcome or actual-flight fields in X.")

No forbidden outcome or actual-flight fields in X.
